# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

The dataset contains structured records about 77 cancer survivors with second primary colorectal cancer, including clinical, pathological, and molecular biomarker variables.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

Below, we enumerate the available record sets and inspect their associated fields, referencing every item by its `@id` as required by the FAIR² Croissant schema.

In [ ]:
# List available record sets by @id
record_sets = list(dataset.record_sets)
print(f"Available record sets (@id):")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '')}")

# For each record set, list available fields and their @id
for rs in record_sets:
    print(f"\nRecord Set '{rs['name'] if 'name' in rs else rs['@id']}' (@id: {rs['@id']}):")
    fields = rs.get('field', [])
    # 'field' can be a dict or list of dicts
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        field_name = fld.get('name', fld.get('@id', ''))
        print(f"    - Field: {field_name} (@id: {fld['@id']})")

# Pick the first available record set for demonstration
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"\nSample record from {example_record_set_id}:")
    # Show a sample record by @id
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i > 0:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview section above.

Below, we extract data from all record sets, keyed by their `@id`, creating a dictionary of DataFrames for convenient access.

In [ ]:
# Extract data from each record set using @id
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading records for record set: {rs_id}")
    try:
        records_lst = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records_lst)
        dataframes[rs_id] = df
        print(f"  - {len(df)} records, fields: {df.columns.tolist()}")
    except Exception as e:
        print(f"  - Failed to load records: {e}")
    print()

# For illustration, display the columns and first few rows for the first record set
example_record_set_id = record_sets[0]['@id'] if record_sets else None
if example_record_set_id:
    print(f"Columns in '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Below, we demonstrate EDA on the primary record set, referencing fields by their `@id`.
Typical steps: filtering records based on criteria, normalizing numeric fields, and grouping data.

> Adjust the `numeric_field_id` and `group_field_id` below to match field `@id`s in the chosen record set.

In [ ]:
# Perform some analysis on the first available record set
primary_rs_id = example_record_set_id
df = dataframes.get(primary_rs_id)
print(f"Analyzing record set: {primary_rs_id}")

# List columns and their @id for reference
print("Available columns (may correspond to field @id or name):")
print(df.columns.tolist())

# Select fields for EDA - you may need to edit field ids based on actual schema info
# As an example, let's use age at 2nd diagnosis as numeric_field, and sex as group field if present
# We'll try several reasonable field id candidates
field_candidates = ['age_at_second_crc_diagnosis', 'age_at_second_diagnosis',
                    'Age_at_second_CRC_diagnosis', 'cr:field/age_at_2nd_dx',
                    'age', 'cr:field/age']
group_candidates = ['sex', 'Sex', 'gender', 'cr:field/sex']

# Find the first matching field for numeric analysis
numeric_field_id = None
for f in field_candidates:
    if f in df.columns:
        numeric_field_id = f
        break
if numeric_field_id is None:
    print("No numeric 'age' field found for analysis. Aborting EDA section.")
else:
    print(f"Using '{numeric_field_id}' as numeric field.")
    threshold = 60
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the field
    # If the field is object dtype, try to convert to numeric
    if filtered_df[numeric_field_id].dtype == object:
        filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df.dropna(subset=[numeric_field_id])

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a suitable group field
    group_field = None
    for gf in group_candidates:
        if gf in df.columns:
            group_field = gf
            break
    if group_field:
        print(f"\nGrouping by '{group_field}'...")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found for grouping.")

## 5. Visualization
Below, we visualize the distribution of the selected numeric field, and (if available) highlight comparisons by group (e.g. sex/gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    # Plot histogram
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=10, color='skyblue')
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No valid data available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a clinical oncology dataset using the Croissant schema and `mlcroissant` library, referencing all entities by their canonical `@id` fields.

Key steps:
- Dataset metadata and record sets referenced by Croissant `@id`
- Tabular extraction by `@id` and EDA on numeric and grouped fields
- Simple visualizations for summary statistics

Further analysis could involve statistical testing, advanced visualization, or building ML models on well-curated fields from the FAIR² dataset.